<a href="https://colab.research.google.com/github/chetanlohia0/Conversion-of-pdf-to-markdown/blob/main/Conversion_of_pdf_to_md_using_the_Unlimited_OCR_baidu_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python --version

import transformers

print("Transformers :", transformers.__version__)

Python 3.12.13
Transformers : 5.12.1


In [ ]:
!pip uninstall -y transformers torch torchvision torchaudio

In [ ]:
!pip install \
torch==2.10.0 \
torchvision==0.25.0 \
transformers==4.57.1 \
accelerate \
sentencepiece \
Pillow==12.1.1 \
matplotlib==3.10.8 \
einops==0.8.2 \
addict==2.4.0 \
easydict==1.13 \
pymupdf==1.27.2.2 \
psutil==7.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/5

In [ ]:
import torch
import transformers

print("=" * 60)
print("Python OK")
print("=" * 60)

print("Transformers :", transformers.__version__)
print("Torch        :", torch.__version__)

print()

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print("Memory :", round(props.total_memory / 1024**3, 2), "GB")

Python OK
Transformers : 4.57.1
Torch        : 2.10.0+cu128

CUDA Available : True
GPU : Tesla T4
Memory : 14.56 GB


model managenent

In [ ]:
import gc
import os
import traceback
from pathlib import Path

import fitz
import torch

from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "baidu/Unlimited-OCR"

MODEL = None
TOKENIZER = None

In [ ]:
def gpu_status():

    if not torch.cuda.is_available():
        print("CUDA unavailable")
        return

    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2

    print(f"Allocated : {allocated:.2f} MB")
    print(f"Reserved  : {reserved:.2f} MB")


def clear_gpu():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.empty_cache()

        torch.cuda.ipc_collect()

    gc.collect()

In [ ]:
gpu_status()

Allocated : 0.00 MB
Reserved  : 0.00 MB


In [ ]:
def load_model():

    global MODEL
    global TOKENIZER

    if MODEL is not None:

        print("Model already loaded.")

        return

    print("Loading model...")

    TOKENIZER = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True
    )

    MODEL = AutoModel.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        use_safetensors=True,
        torch_dtype=torch.bfloat16,
    )

    MODEL = MODEL.eval().cuda()

    print("Model loaded.")

    gpu_status()


def destroy_model():

    global MODEL
    global TOKENIZER

    print("Destroying model...")

    if MODEL is not None:

        del MODEL

    if TOKENIZER is not None:

        del TOKENIZER

    MODEL = None
    TOKENIZER = None

    clear_gpu()

    print("Model destroyed.")

    gpu_status()

run these to check the model and gpu allocation

In [ ]:
load_model()

In [ ]:
gpu_status()

Allocated : 9.12 MB
Reserved  : 22.00 MB


In [ ]:
destroy_model()

Destroying model...
Model destroyed.
Allocated : 9.12 MB
Reserved  : 22.00 MB


In [ ]:
gpu_status()

Allocated : 9.12 MB
Reserved  : 22.00 MB


till here


In [ ]:
#file upload test and conversion to pages

from google.colab import files
import tempfile

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No PDF uploaded.")

PDF_PATH = list(uploaded.keys())[0]

PDF_NAME = Path(PDF_PATH).stem

OUTPUT_ROOT = Path("outputs") / PDF_NAME

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("PDF :", PDF_PATH)
print("Output :", OUTPUT_ROOT)



# pdf to image test



Saving APAC 2024 claims file layout Appendices A-G.pdf to APAC 2024 claims file layout Appendices A-G.pdf
PDF : APAC 2024 claims file layout Appendices A-G.pdf
Output : outputs/APAC 2024 claims file layout Appendices A-G


ocr ingine

In [ ]:
# tesing of creation of batches

BATCH_SIZE = 4

BATCHES = []

batch_id = 1

for start in range(0, len(PAGES), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(PAGES))

    BATCHES.append({

        "id": batch_id,

        "start": start + 1,

        "end": end,

        "pages": PAGES[start:end]

    })

    batch_id += 1

print(f"Total Batches : {len(BATCHES)}")

print()

for batch in BATCHES[:3]:

    print(batch)

Total Batches : 11

{'id': 1, 'start': 1, 'end': 4, 'pages': ['/tmp/pdf_pages_ja7qam59/page_0001.png', '/tmp/pdf_pages_ja7qam59/page_0002.png', '/tmp/pdf_pages_ja7qam59/page_0003.png', '/tmp/pdf_pages_ja7qam59/page_0004.png']}
{'id': 2, 'start': 5, 'end': 8, 'pages': ['/tmp/pdf_pages_ja7qam59/page_0005.png', '/tmp/pdf_pages_ja7qam59/page_0006.png', '/tmp/pdf_pages_ja7qam59/page_0007.png', '/tmp/pdf_pages_ja7qam59/page_0008.png']}
{'id': 3, 'start': 9, 'end': 12, 'pages': ['/tmp/pdf_pages_ja7qam59/page_0009.png', '/tmp/pdf_pages_ja7qam59/page_0010.png', '/tmp/pdf_pages_ja7qam59/page_0011.png', '/tmp/pdf_pages_ja7qam59/page_0012.png']}


In [ ]:
def pdf_to_images(pdf_path, dpi=300):

    doc = fitz.open(pdf_path)

    temp_dir = Path(
        tempfile.mkdtemp(prefix="pdf_pages_")
    )

    pages = []

    matrix = fitz.Matrix(dpi / 72, dpi / 72)

    for page_number, page in enumerate(doc):

        img_path = temp_dir / f"page_{page_number+1:04d}.png"

        page.get_pixmap(matrix=matrix).save(str(img_path))

        pages.append(str(img_path))

    doc.close()

    return pages


In [ ]:
from pathlib import Path

def batch_output_folder(batch):

    folder = OUTPUT_ROOT / (

        f"batch_{batch['id']}"

        f"_pages_{batch['start']:03d}_{batch['end']:03d}"

    )

    folder.mkdir(

        parents=True,

        exist_ok=True

    )

    return folder

In [ ]:
# testing
folder = batch_output_folder(BATCHES[0])

print(folder)

outputs/APAC 2024 claims file layout Appendices A-G/batch_001_pages_001_004


In [ ]:
from pathlib import Path

def validate_output(out_dir, expected_pages):

    md_file = out_dir / "result.md"

    if not md_file.exists():
        return False, "Missing result.md"

    text = md_file.read_text(
        encoding="utf8",
        errors="ignore"
    )

    # Empty file
    if len(text.strip()) < 100:
        return False, "Output too small"

    # Page count
    page_count = text.count("<PAGE>")

    if page_count != expected_pages:
        return False, (
            f"Expected {expected_pages} pages "
            f"but found {page_count}"
        )

    # OCR extracted almost nothing
    word_count = len(text.split())

    if word_count < expected_pages * 30:
        return False, (
            f"Too few words ({word_count})"
        )

    # Optional quality indicators
    has_table = "<table>" in text
    has_non_text = "[Non-Text]" in text

    if not has_table and not has_non_text:
        print("Warning: No tables or non-text regions detected.")

    return True, "OK"

In [ ]:
SUCCESS = "SUCCESS"

OOM = "OOM"

ERROR = "ERROR"

INVALID = "INVALID"

SKIPPED = "SKIPPED"

In [ ]:
from datetime import datetime

def log_batch(batch, status, message=""):

    out_dir = batch_output_folder(batch)

    log_file = out_dir / "batch.log"

    with open(log_file, "a", encoding="utf8") as f:

        f.write("=" * 60 + "\n")

        f.write(f"Time   : {datetime.now()}\n")

        f.write(f"Batch  : {batch['id']}\n")

        f.write(
            f"Pages  : {batch['start']} - {batch['end']}\n"
        )

        f.write(f"Status : {status}\n")

        if message:

            f.write(f"Message: {message}\n")

        f.write("\n")

In [ ]:
import traceback

def process_batch(batch):

    out_dir = batch_output_folder(batch)

    done_file = out_dir / ".done"

    if done_file.exists():

        print(f"✓ Batch {batch['id']} already completed.")

        return SKIPPED

    print("\n" + "="*70)
    print(
        f"Batch {batch['id']} "
        f"({batch['start']} -> {batch['end']})"
    )
    print("="*70)

    try:

        MODEL.infer_multi(

            TOKENIZER,

            prompt="<image>Multi page parsing.",

            image_files=batch["pages"],

            output_path=str(out_dir),

            image_size=1024,

            max_length=32768,

            no_repeat_ngram_size=35,

            ngram_window=1024,

            save_results=True,

        )

    except torch.OutOfMemoryError:

        clear_gpu()

        log_batch(
            batch,
            OOM,
            "CUDA Out Of Memory"
        )

        return OOM


    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            clear_gpu()

            log_batch(
                batch,
                OOM,
                str(e)
            )

            return OOM

        traceback.print_exc()

        clear_gpu()

        log_batch(
            batch,
            ERROR,
            str(e)
        )

        return ERROR


    except Exception as e:

        traceback.print_exc()

        clear_gpu()

        log_batch(
            batch,
            ERROR,
            str(e)
        )

        return ERROR


    clear_gpu()

    valid, reason = validate_output(

        out_dir,

        len(batch["pages"])

    )

    if not valid:

        print(reason)

        log_batch(
            batch,
            INVALID,
            reason
        )

        return INVALID


    log_batch(
        batch,
        SUCCESS
    )

    done_file.touch()

    print("✓ Batch completed.")

    return SUCCESS

In [ ]:
# test for run of first batch

load_model()

status = process_batch(BATCHES[0])

print(status)


Model already loaded.
✓ Batch 1 already completed.
SKIPPED


In [ ]:
# can delete this

def retry_batch(batch):

    # -------------------------------
    # First Attempt
    # -------------------------------

    status = process_batch(batch)

    if status in [SUCCESS, SKIPPED]:
        return status

    # -------------------------------
    # GPU Cleanup Retry
    # -------------------------------

    if status == OOM:

        print("\nOOM detected.")
        print("Cleaning GPU and retrying...")

        clear_gpu()

        status = process_batch(batch)

        if status in [SUCCESS, SKIPPED]:
            return status

    # -------------------------------
    # Reload Model Retry
    # -------------------------------

    if status == OOM:

        print("\nReloading model...")

        destroy_model()

        load_model()

        status = process_batch(batch)

        if status in [SUCCESS, SKIPPED]:
            return status

    # -------------------------------
    # Batch Failed
    # -------------------------------

    out_dir = batch_output_folder(batch)

    (out_dir / ".failed").touch()

    print(f"❌ Batch {batch['id']} Failed.")

    return status

In [ ]:
def split_batch(batch):

    pages = batch["pages"]

    if len(pages) == 1:
        return []

    mid = len(pages) // 2

    left_pages = pages[:mid]
    right_pages = pages[mid:]

    left = {

        "id": batch["id"] + "A",

        "start": batch["start"],

        "end": batch["start"] + len(left_pages) - 1,

        "pages": left_pages

    }

    right = {

        "id": batch["id"] + "B",

        "start": left["end"] + 1,

        "end": batch["end"],

        "pages": right_pages

    }

    return [left, right]

In [ ]:
FAILED_BATCHES = []

def process_failed_batches():

    global FAILED_BATCHES

    while FAILED_BATCHES:

        batch = FAILED_BATCHES.pop(0)

        print("\n" + "="*70)
        print(f"Recovering Batch {batch['id']}")
        print(
            f"Pages : {batch['start']} -> {batch['end']}"
        )
        print(
            f"Remaining Queue : {len(FAILED_BATCHES)}"
        )
        print("="*70)

        children = split_batch(batch)

        if len(children) == 0:

            print("Single page still failed.")

            out = batch_output_folder(batch)

            (out / ".failed").touch()

            continue


        for child in children:

            status = process_batch(child)

            clear_gpu()

            if status in [SUCCESS, SKIPPED]:

                continue

            FAILED_BATCHES.append(child)

In [ ]:
def document_statistics():

    merged = OUTPUT_ROOT / "merged.md"

    text = merged.read_text(
        encoding="utf8",
        errors="ignore"
    )

    print("="*60)

    print("Document Statistics")

    print("="*60)

    print("Pages :", text.count("<PAGE>"))

    print("Tables :", text.count("<table>"))

    print("Non Text :", text.count("[Non-Text]"))

    print("Words :", len(text.split()))

    print("Characters :", len(text))

In [ ]:
from google.colab import files
import tempfile
import fitz
import os
from pathlib import Path

while True:

    print("\n" + "="*80)
    print("UPLOAD PDF")
    print("="*80)

    uploaded = files.upload()

    if len(uploaded) == 0:
        print("No file uploaded.")
        break

    pdf_path = list(uploaded.keys())[0]

    pdf_name = Path(pdf_path).stem

    OUTPUT_ROOT = Path("outputs") / pdf_name
    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    print(f"\nPDF : {pdf_name}")

    # --------------------------------------------------
    # Convert PDF -> Images
    # --------------------------------------------------

    PAGES = pdf_to_images(pdf_path)

    print(f"Total Pages : {len(PAGES)}")

    # --------------------------------------------------
    # Create Batches
    # --------------------------------------------------


    BATCH_SIZE = 4

    BATCHES = []

    batch_no = 1

    for start in range(0, len(PAGES), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(PAGES))

        BATCHES.append({

            "id": f"B{batch_no:03d}",

            "start": start + 1,

            "end": end,

            "pages": PAGES[start:end]

        })

        batch_no += 1

    print(f"Total Batches : {len(BATCHES)}")


    # --------------------------------------------------
    # Load Model
    # --------------------------------------------------

    clear_gpu()

    load_model()

    # --------------------------------------------------
    # First Extraction Pass
    # --------------------------------------------------

    FAILED_BATCHES = []

    SUCCESS_BATCHES = []

    print("\n")
    print("="*80)
    print("FIRST PASS")
    print("="*80)

    for batch in BATCHES:

        status = process_batch(batch)

        if status in [SUCCESS, SKIPPED]:

            SUCCESS_BATCHES.append(batch)

        else:

            FAILED_BATCHES.append(batch)

    print()

    print("="*80)
    print("FIRST PASS SUMMARY")
    print("="*80)

    print("Successful :", len(SUCCESS_BATCHES))
    print("Failed     :", len(FAILED_BATCHES))

    # --------------------------------------------------
    # Recovery
    # --------------------------------------------------



UPLOAD PDF


Saving APAC 2024 claims file layout Appendices A-G.pdf to APAC 2024 claims file layout Appendices A-G.pdf

PDF : APAC 2024 claims file layout Appendices A-G
MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>header [42, 56, 203, 135]<|/det|>Oregon Health Authority
<|det|>header [219, 56, 258, 135]<|/det|>[Non-Text]
<|det|>header [264, 56, 435, 135]<|/det|>APAC
ALL PAYER • ALL CLAIMS
<|det|>header [815, 123, 921, 145]<|/det|>Version 2024.0
<|det|>title [65, 145, 329, 168]<|/det|>b. Claims file control totals example
<|det|>text [87, 181, 393, 205]<|/det|>Example when all file types are submitted
<|det|>table [175, 228, 781, 398]<|/det|><table><tr><td>Payer</td><td>File</td><td>Data_Rows</td><td>Amt_Billed</td><td>Amt_Paid</td></tr><tr><td>OMIP</td><td>Medical</td><td>12345678</td><td>123456789.12</td><td>123456789.12</td></tr><tr><td>OMIP</td><td>Pharmacy</td><td>12345678</td><td>123456789.12</td><td>123456789.12</td></tr><tr><td>OMIP</td><td>Enrollment</td><td>12345678</td><td></td><td></td></tr><tr><td>OMIP</td><td>Provider</td><td>123456</td><td></td><td></td></tr><tr><td>OMIP</td><td>Premium</td><td>12345</td><td>123456789.12</td><td></td></tr><tr><td>OMIP</td><td>Dental</t

image_page_0: 0it [00:00, ?it/s]
other_page_0: 100%|██████████| 20/20 [00:00<00:00, 78398.21it/s]
image_page_1: 0it [00:00, ?it/s]
other_page_1: 100%|██████████| 9/9 [00:00<00:00, 68509.50it/s]
image_page_2: 0it [00:00, ?it/s]
other_page_2: 100%|██████████| 18/18 [00:00<00:00, 91957.94it/s]


✓ Batch completed.

FIRST PASS SUMMARY
Successful : 10
Failed     : 1

UPLOAD PDF


No file uploaded.


In [ ]:
if FAILED_BATCHES:

    print()
    print("=" * 80)
    print("RECOVERY PHASE")
    print("=" * 80)

    process_failed_batches()

print()
print("=" * 80)
print("RECOVERY COMPLETE")
print("=" * 80)

print("Remaining Failed :", len(FAILED_BATCHES))




RECOVERY PHASE

Recovering Batch B007
Pages : 25 -> 28
Remaining Queue : 0

Batch B007A (25 -> 26)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>header [41, 54, 201, 135]<|/det|>Oregon Health Authority
<|det|>header [242, 54, 270, 117]<|/det|>[Non-Text]
<|det|>header [275, 54, 430, 117]<|/det|>APAC
<|det|>header [288, 111, 430, 133]<|/det|>ALL PAYER • ALL CLAIMS
<|det|>header [816, 123, 919, 144]<|/det|>Version 2024.0
<|det|>table [41, 144, 957, 885]<|/det|><table><tr><td>Data element</td><td>Name</td><td>Max. length</td><td>Type</td><td>Required?</td><td>NCPDP Field</td><td>NCPDP Source</td><td>Description/valid values</td><td>Error threshold</td></tr><tr><td>PC026</td><td>NDC</td><td>11</td><td>Text</td><td>Yes</td><td>407-D7</td><td>C</td><td>National Drug Code (NDC)</td><td>1.2%</td></tr><tr><td>PC032</td><td>Date filled</td><td>8</td><td>Date</td><td>Yes</td><td>401-D1</td><td>C</td><td>Date the prescription was filled. CCYYMMDD (example: 20090624)</td><td>0.0%</td></tr><tr><td>PC017</td><td>Payment date</td><td>8</td><td>Date</td><td>Situational</td><td>216</td><td>P</td><td>CCYYMMDD (example: 20090624). Blan

image_page_0: 0it [00:00, ?it/s]
other_page_0: 100%|██████████| 13/13 [00:00<00:00, 77122.99it/s]


Expected 2 pages but found 1

Batch B007B (27 -> 28)


...

✓ Batch completed.


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.



Batch B007AB (26 -> 26)
<PAGE><|det|>header [42, 56, 201, 130]<|/det|>Oregon Health Authority
<|det|>header [242, 56, 280, 113]<|/det|>[Non-Text]
<|det|>header [291, 56, 430, 113]<|/det|>APAC
<|det|>header [290, 113, 430, 135]<|/det|>ALL PAYER • ALL CLAIMS
<|det|>header [816, 123, 916, 144]<|/det|>Version 2024.0
<|det|>table [42, 145, 957, 812]<|/det|><table><tr><td>Data element</td><td>Name</td><td>Max. length</td><td>Type</td><td>Required?</td><td>NCPDP Field</td><td>NCPDP Source</td><td>Description/valid values</td><td>Error threshold</td></tr><tr><td>PC039</td><td>Dispensing fee paid</td><td>12</td><td>Numeric</td><td>Yes</td><td>506-F7</td><td>C</td><td>Two explicit decimal places. Enter 0 if amount equals zero. Leave blank if missing. Example: 15102.00</td><td>0.0%</td></tr><tr><td>PC040</td><td>Co-pay applied</td><td>12</td><td>Numeric</td><td>Yes</td><td>518-F1</td><td>C</td><td>Two explicit decimal places. Enter 0 if amount equals zero. Leave blank if missing. Example: 15102.

image_page_0: 0it [00:00, ?it/s]
other_page_0: 100%|██████████| 12/12 [00:00<00:00, 69422.96it/s]


✓ Batch completed.

RECOVERY COMPLETE
Remaining Failed : 0


In [ ]:
from pathlib import Path
import re

def merge_markdown():

    batch_dirs = list(OUTPUT_ROOT.glob("batch_*"))

    print(f"\nFound {len(batch_dirs)} batch folders")

    batches = []

    # -----------------------------------
    # Read metadata
    # -----------------------------------

    for folder in batch_dirs:

        log_file = folder / "batch.log"
        done_file = folder / ".done"
        md_file = folder / "result.md"

        if not done_file.exists():
            continue

        if not md_file.exists():
            continue

        m = re.search(
            r"batch_(B[0-9A-Z]+)_pages_(\d+)_(\d+)",
            folder.name
        )

        if m is None:
            continue

        batch_id = m.group(1)
        start = int(m.group(2))
        end = int(m.group(3))

        batches.append({

            "id": batch_id,
            "start": start,
            "end": end,
            "folder": folder

        })

    # -----------------------------------
    # Remove parent batches
    # -----------------------------------

    leaf_batches = []

    ids = [b["id"] for b in batches]

    for batch in batches:

        is_parent = False

        for other in ids:

            if other.startswith(batch["id"]) and other != batch["id"]:

                is_parent = True
                break

        if not is_parent:
            leaf_batches.append(batch)

    # -----------------------------------
    # Sort by page
    # -----------------------------------

    leaf_batches.sort(
        key=lambda x: x["start"]
    )

    print()

    print("Leaf batches used for merge:")

    for b in leaf_batches:

        print(
            f"{b['id']} : "
            f"{b['start']} -> {b['end']}"
        )

    # -----------------------------------
    # Merge
    # -----------------------------------

    merged = []

    for batch in leaf_batches:

        md = batch["folder"] / "result.md"

        with open(
            md,
            "r",
            encoding="utf8",
            errors="ignore"
        ) as f:

            merged.append(
                f.read().strip()
            )

    merged_file = OUTPUT_ROOT / "merged.md"

    with open(
        merged_file,
        "w",
        encoding="utf8"
    ) as f:

        f.write("\n\n".join(merged))

    print()

    print("Merged File Saved")

    print(merged_file)

    return merged_file

In [ ]:
merged_md = merge_markdown()


Found 15 batch folders

Leaf batches used for merge:
B001 : 1 -> 4
B002 : 5 -> 8
B003 : 9 -> 12
B004 : 13 -> 16
B005 : 17 -> 20
B006 : 21 -> 24
B007AA : 25 -> 25
B007AB : 26 -> 26
B007B : 27 -> 28
B008 : 29 -> 32
B009 : 33 -> 36
B010 : 37 -> 40
B011 : 41 -> 43

Merged File Saved
outputs/APAC 2024 claims file layout Appendices A-G/merged.md


In [ ]:
document_statistics()

Document Statistics
Pages : 43
Tables : 48
Non Text : 31
Words : 6780
Characters : 90905
